<a href="https://colab.research.google.com/github/mohamedelguendy/Secure-Banking-System-Simulation/blob/main/Secure_Banking_System_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import re
import getpass

# =========================
# DATABASE
# =========================
users = {}
global_logs = []
current_user = None

MAX_ATTEMPTS = 5


# =========================
# PASSWORD STRENGTH CHECKER
# =========================
def is_strong_password(password):
    return (
        len(password) >= 6 and
        re.search(r"[A-Z]", password) and
        re.search(r"[0-9]", password)
    )


# =========================
# REGISTER USER (FIXED FLOW)
# =========================
def register():
    print("\n=== REGISTER USER ===")

    while True:
        username = input("Username (or type 'exit'): ")

        if username.lower() == "exit":
            return

        if username in users:
            print("User already exists!")
            continue

        password = getpass.getpass("Password: ")

        if not is_strong_password(password):
            print("Weak password! Must contain:")
            print("- At least 6 characters")
            print("- 1 uppercase letter")
            print("- 1 number")
            continue  # stays in register loop

        users[username] = {
            "password": password,
            "balance": 0,
            "history": [],
            "failed_attempts": 0,
            "locked": False
        }

        print("User created successfully!")
        break


# =========================
# LOGIN SYSTEM (5 ATTEMPTS)
# =========================
def login():
    global current_user

    print("\n=== LOGIN ===")

    attempts_left = MAX_ATTEMPTS

    while attempts_left > 0:

        username = input("Username (or type 'exit'): ")

        if username.lower() == "exit":
            return False

        password = getpass.getpass("Password: ")

        if username not in users:
            print("User not found.")
            attempts_left -= 1
            print(f"Attempts left: {attempts_left}")
            continue

        user = users[username]

        if user["locked"]:
            print("Account is locked.")
            return False

        if password == user["password"]:
            print("Login successful!")
            user["failed_attempts"] = 0
            current_user = username
            return True

        # wrong password
        attempts_left -= 1
        user["failed_attempts"] += 1

        print("Wrong password!")
        print(f"Attempts left: {attempts_left}")

        if user["failed_attempts"] >= MAX_ATTEMPTS:
            user["locked"] = True
            print("Account locked due to too many failed attempts.")
            return False

    print("Login failed.")
    return False


# =========================
# DEPOSIT MONEY
# =========================
def deposit():
    amount = float(input("Deposit amount: "))

    if amount <= 0:
        print("Invalid amount.")
        return

    users[current_user]["balance"] += amount

    log = f"{current_user} deposited {amount}"
    users[current_user]["history"].append(log)
    global_logs.append(log)

    print("Deposit successful!")


# =========================
# TRANSFER MONEY
# =========================
def transfer():
    receiver = input("Receiver: ")
    amount = float(input("Amount: "))

    if receiver not in users:
        print("Receiver not found.")
        return

    if amount <= 0:
        print("Invalid amount.")
        return

    if users[current_user]["balance"] < amount:
        print("Insufficient balance.")
        return

    users[current_user]["balance"] -= amount
    users[receiver]["balance"] += amount

    sender_log = f"Sent {amount} to {receiver}"
    receiver_log = f"Received {amount} from {current_user}"

    users[current_user]["history"].append(sender_log)
    users[receiver]["history"].append(receiver_log)

    global_logs.append(sender_log)
    global_logs.append(receiver_log)

    print("Transfer successful!")


# =========================
# VIEW HISTORY
# =========================
def view_history():
    print("\n=== TRANSACTION HISTORY ===")

    history = users[current_user]["history"]

    if not history:
        print("No transactions.")
        return

    for h in history:
        print("-", h)


# =========================
# CHANGE PASSWORD (FIXED)
# =========================
def change_password():
    print("\n=== CHANGE PASSWORD ===")

    old = getpass.getpass("Old password: ")

    if old != users[current_user]["password"]:
        print("Incorrect password.")
        return

    while True:
        new = getpass.getpass("New password: ")

        if not is_strong_password(new):
            print("Weak password!")
            continue

        confirm = getpass.getpass("Confirm password: ")

        if new != confirm:
            print("Passwords do not match!")
            continue

        users[current_user]["password"] = new
        print("Password changed successfully!")
        break


# =========================
# USER MENU
# =========================
def user_menu():
    while True:
        print("\n=== USER MENU ===")
        print("1. Balance")
        print("2. Deposit")
        print("3. Transfer")
        print("4. History")
        print("5. Change Password")
        print("6. Logout")

        choice = input("Choose: ")

        if choice == "1":
            print("Balance:", users[current_user]["balance"])
        elif choice == "2":
            deposit()
        elif choice == "3":
            transfer()
        elif choice == "4":
            view_history()
        elif choice == "5":
            change_password()
        elif choice == "6":
            break
        else:
            print("Invalid option!")


# =========================
# MAIN MENU
# =========================
def main():
    while True:
        print("\n=== BANK SYSTEM ===")
        print("1. Register")
        print("2. Login")
        print("3. View Logs")
        print("4. Exit")

        choice = input("Choose: ")

        if choice == "1":
            register()

        elif choice == "2":
            if login():
                user_menu()

        elif choice == "3":
            print("\n=== GLOBAL LOGS ===")
            for log in global_logs:
                print("-", log)

        elif choice == "4":
            print("Exiting system...")
            break

        else:
            print("Invalid choice!")


print("=== SIMPLE BANKING SYSTEM ===")
main()

=== SIMPLE BANKING SYSTEM ===

=== BANK SYSTEM ===
1. Register
2. Login
3. View Logs
4. Exit
Choose: 1

=== REGISTER USER ===
Username (or type 'exit'): mo
Password: ··········
User created successfully!

=== BANK SYSTEM ===
1. Register
2. Login
3. View Logs
4. Exit
Choose: 2

=== LOGIN ===
Username (or type 'exit'): mo
Password: ··········
Login successful!

=== USER MENU ===
1. Balance
2. Deposit
3. Transfer
4. History
5. Change Password
6. Logout
Choose: 2
Deposit amount: 10000
Deposit successful!

=== USER MENU ===
1. Balance
2. Deposit
3. Transfer
4. History
5. Change Password
6. Logout
Choose: 3
Receiver: wa12345
Amount: 1000
Receiver not found.

=== USER MENU ===
1. Balance
2. Deposit
3. Transfer
4. History
5. Change Password
6. Logout
Choose: 4

=== TRANSACTION HISTORY ===
- mo deposited 10000.0

=== USER MENU ===
1. Balance
2. Deposit
3. Transfer
4. History
5. Change Password
6. Logout
Choose: 6

=== BANK SYSTEM ===
1. Register
2. Login
3. View Logs
4. Exit
Choose: 3

=== GLOB